# Dynamic Prompt
<img src="./assets/LC_DynamicPrompts.png" width="500">

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env(".env")

OPENAI_API_KEY=****here
LANGSMITH_API_KEY=****2126
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    is_employee: bool
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    is_employee = runtime.context.is_employee
    db = runtime.context.db

    # 1. Define allowed tables for non-employees
    allowed_tables = {"Album", "Artist", "Genre", "Playlist", "PlaylistTrack", "Track"}
    
    # 2. Hard Check for non-employees
    if not is_employee:
        restricted_tables = {"Customer", "Invoice", "Employee", "InvoiceLine"}
        for table in restricted_tables:
            if table.upper() in query.upper():
                return f"Error: Access Denied. You do not have permission to access the '{table}' table."

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
# Run this cell first
schema = db.get_table_info()

SYSTEM_PROMPT_TEMPLATE = f"""You are a careful SQLite analyst.

Here is the database schema:
{schema}

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only; no INSERT/UPDATE/DELETE.
- Limit to 5 rows unless asked otherwise.
{{table_limits}}
- If the tool returns 'Error:', revise the SQL and try again.
- To find a customer's purchase, look for the 'Customer' and 'Invoice' tables.
"""

## Build a Dynamic Prompt
Utilize runtime context and middleware to generate a dynamic prompt.

In [6]:
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt

@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    current_schema = db.get_table_info()
    context = request.runtime.context
    
    if not getattr(context, "is_employee", False):
        # Explicitly tell the LLM to REFUSE questions about these tables
        table_limits = (
            "CRITICAL: You are strictly forbidden from accessing 'Customer', 'Invoice', or 'Employee' tables. "
            "If a user asks about purchases, names, or private data, explain that you do not have permission."
        )
    else:
        table_limits = "You have full administrative access."

    return SYSTEM_PROMPT_TEMPLATE.format(table_limits=table_limits)

Include middleware in `create_agent`.

In [7]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:14b", temperature=0)

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[execute_sql],
    middleware=[dynamic_system_prompt],
    context_schema=RuntimeContext,
)

In [9]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=False, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================

I don't have access to customer purchase data or the ability to look up information involving the Customer, Invoice, or Employee tables. This request would require accessing restricted private data.


In [10]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=True, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (aa22f5cf-9bb9-42c5-bc9f-29938e977268)
 Call ID: aa22f5cf-9bb9-42c5-bc9f-29938e977268
  Args:
    query: SELECT CustomerId FROM Customer WHERE FirstName = 'Frank' AND LastName = 'Harris';
================================= Tool Message =================================
Name: execute_sql

[(16,)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (820142c7-e0cc-400d-b756-ef0e4ee725a8)
 Call ID: 820142c7-e0cc-400d-b756-ef0e4ee725a8
  Args:
    query: SELECT Total FROM Invoice WHERE CustomerId = 16 ORDER BY Total DESC LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

[(13.86,)]
================================== Ai Message ===================